# 012. Final — 한국어 챗봇 Transformer 학습 & 저장

00~05주차에서 직접 만든 PositionalEncoding, MultiHeadAttention, Encoder, Decoder, Mask 함수를 이어받아
**완전한 Encoder-Decoder Transformer**를 조립하고, 실제 한국어 대화 데이터로 학습시켜 체크포인트로 저장한다.

저장된 체크포인트는 `final.py`(Streamlit)가 불러와 실시간 채팅 + 간단한 하드코딩 Function Calling(Agent)로 서빙한다.

> 00~05주차와 동일하게, 이 노트북도 모델 코드를 별도 모듈로 분리하지 않고 전부 이 노트북 안에 직접 담는다.
> `final.py`는 같은 모델 클래스를 자기 파일 안에 그대로 복사해서 갖고 있다 — 체크포인트(state_dict)를 불러오려면
> 클래스 구조가 정확히 같아야 하므로, 이 노트북에서 모델 구조를 바꾸면 `final.py` 쪽도 같이 맞춰야 한다.


In [1]:
# Colab/로컬 어디서 실행하든 필요한 패키지를 설치 (이미 있으면 바로 스킵됨)
%pip install -q torch pandas scikit-learn tqdm

In [2]:
import math

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# GPU 사용 가능하면 GPU, 아니면 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### 0. 이전 주차 부품 다시 가져오기

00~05주차에서 만든 Mask, PositionalEncoding, Attention, Encoder, Decoder를 이 노트북에도 그대로 다시 적는다
(00~05주차와 마찬가지로, 노트북마다 필요한 코드를 직접 담는 스타일을 따른다).


In [3]:
def make_padding_mask(seq, pad_idx=0):
    # seq: [B, T] 토큰 id -> [B, 1, 1, T], <pad> 위치만 False
    return (seq != pad_idx).unsqueeze(1).unsqueeze(2)


def make_causal_mask(seq_len):
    # 대각선 포함 아래쪽(과거+현재)만 True인 하삼각행렬 -> [T, T]
    return torch.tril(torch.ones(seq_len, seq_len)).bool()


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T]


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    output = weights @ V

    return output, weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.h = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, _ = x.shape
        x = x.view(B, T, self.h, self.d_k)
        return x.transpose(1, 2)  # [B, h, T, d_k]

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.split_heads(self.W_q(query))
        K = self.split_heads(self.W_k(key))
        V = self.split_heads(self.W_v(value))

        out, _ = scaled_dot_product_attention(Q, K, V, mask)

        out = out.transpose(1, 2).contiguous().view(B, -1, self.h * self.d_k)
        return self.W_o(out)


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


In [4]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out = self.mha(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))

        return x


class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, causal_mask=None, pad_mask=None):
        # 1) Masked Self-Attention: 미래를 가리는 causal_mask(+padding)
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, causal_mask)))

        # 2) Cross-Attention: query=Decoder(x), key=value=Encoder 출력(enc_out)
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, pad_mask)))

        # 3) Feed Forward
        x = self.norm3(x + self.dropout(self.ffn(x)))

        return x


class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

    def forward(self, x, enc_out, causal_mask=None, pad_mask=None):
        for layer in self.layers:
            x = layer(x, enc_out, causal_mask, pad_mask)
        return x


### 0-1. 챗봇 Transformer 조립

Encoder와 Decoder를 합쳐 완전한 Encoder-Decoder Transformer로 만든다.


In [5]:
class ChatbotTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, num_layers=3, num_heads=8,
                 d_ff=512, max_len=64, dropout=0.1, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx

        self.embed_layer = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pe_layer = PositionalEncoding(d_model, max_len=max_len)
        self.dropout = nn.Dropout(dropout)

        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff, dropout)
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff, dropout)

        self.output_linear = nn.Linear(d_model, vocab_size)

    def encode(self, src):
        src_pad_mask = make_padding_mask(src, self.pad_idx)
        x = self.dropout(self.pe_layer(self.embed_layer(src)))
        enc_out = self.encoder(x, src_pad_mask)
        return enc_out, src_pad_mask

    def decode(self, tgt, enc_out, src_pad_mask):
        T = tgt.size(1)
        # causal_mask[T,T] & tgt_pad_mask[B,1,1,T] -> 브로드캐스팅으로 [B,1,T,T]
        causal_mask = make_causal_mask(T).to(tgt.device)
        tgt_pad_mask = make_padding_mask(tgt, self.pad_idx)
        combined_mask = causal_mask & tgt_pad_mask

        x = self.dropout(self.pe_layer(self.embed_layer(tgt)))
        dec_out = self.decoder(x, enc_out, combined_mask, src_pad_mask)

        return self.output_linear(dec_out)

    def forward(self, src, tgt):
        enc_out, src_pad_mask = self.encode(src)
        return self.decode(tgt, enc_out, src_pad_mask)


### 0-2. Tokenizer / Dataset / 추론 함수

01/05주차와 동일한 word-level 토크나이저(`<pad>/<sos>/<eos>/<unk>`), 학습용 Dataset, greedy decoding 추론 함수.


In [6]:
SPECIAL_TOKENS = ("<pad>", "<sos>", "<eos>", "<unk>")


def create_vocab(sentences, special_tokens=SPECIAL_TOKENS):
    words = sorted(set(w for s in sentences for w in s.split()))
    vocab = list(special_tokens) + words
    stoi = {w: i for i, w in enumerate(vocab)}
    itos = {i: w for i, w in enumerate(vocab)}
    return vocab, stoi, itos


def encode(sentence, stoi):
    ids = [stoi.get(w, stoi["<unk>"]) for w in sentence.split()]
    return [stoi["<sos>"]] + ids + [stoi["<eos>"]]


def decode(ids, itos):
    specials = {"<pad>", "<sos>", "<eos>"}
    words = []
    for i in ids:
        w = itos.get(int(i), "<unk>")
        if w == "<eos>":
            break
        if w in specials:
            continue
        words.append(w)
    return " ".join(words)


def pad_ids(ids, max_len, pad_idx=0):
    ids = ids[:max_len]
    return ids + [pad_idx] * (max_len - len(ids))


class ChatbotDataset(Dataset):
    def __init__(self, questions, answers, stoi, max_len=30):
        self.questions = questions
        self.answers = answers
        self.stoi = stoi
        self.max_len = max_len

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        src_ids = pad_ids(encode(self.questions[idx], self.stoi), self.max_len)

        # <sos> A <eos>를 max_len+1로 만든 뒤 한 칸씩 어긋나게 잘라서 teacher forcing에 사용
        tgt_full = pad_ids(encode(self.answers[idx], self.stoi), self.max_len + 1)
        tgt_in = tgt_full[:-1]
        tgt_out = tgt_full[1:]

        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_in, dtype=torch.long),
            torch.tensor(tgt_out, dtype=torch.long),
        )


def generate_reply(model, question, stoi, itos, max_len=30, device="cpu"):
    model.eval()
    src_ids = pad_ids(encode(question, stoi), max_len)
    src = torch.tensor([src_ids], dtype=torch.long, device=device)

    with torch.no_grad():
        enc_out, src_pad_mask = model.encode(src)

        tgt_ids = [stoi["<sos>"]]
        for _ in range(max_len):
            tgt = torch.tensor([tgt_ids], dtype=torch.long, device=device)
            logits = model.decode(tgt, enc_out, src_pad_mask)
            next_id = logits[0, -1].argmax(-1).item()

            if next_id == stoi["<eos>"]:
                break
            tgt_ids.append(next_id)

    reply = decode(tgt_ids[1:], itos)
    return reply if reply else "음... 잘 모르겠어요."


### 1. 데이터 다운로드

[songys/Chatbot_data](https://github.com/songys/Chatbot_data) — 일상 대화 한국어 챗봇 Q&A 11,823쌍
(label: 0 일상, 1 이별, 2 사랑). 05주차에서 NSMC를 `pd.read_csv(url)`로 바로 읽어온 것과 동일한 방식으로,
따로 저장하지 않고 노트북 실행 시점에 직접 다운로드해서 사용한다.


In [7]:
data_url = "https://raw.githubusercontent.com/songys/Chatbot_data/master/ChatbotData.csv"
df = pd.read_csv(data_url)

# NAN값 제거
df = df.dropna(subset=["Q", "A"])
print(df.shape)
df.head()


(11823, 3)


,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


In [8]:
questions, answers, labels = df["Q"].tolist(), df["A"].tolist(), df["label"].tolist()

train_q, test_q, train_a, test_a = train_test_split(
    questions, answers, stratify=labels, test_size=0.1, random_state=42
)
print(len(train_q), len(test_q))


10640 1183


### 2. Vocab 생성

01/05주차와 동일하게 공백 기준 단어 단위 vocab을 만든다. Q와 A에 등장하는 모든 단어를 합쳐서 하나의 vocab을 공유한다
(Encoder 입력과 Decoder 입출력이 같은 임베딩 공간을 쓰기 때문).


In [9]:
vocab, stoi, itos = create_vocab(train_q + train_a)
print(f"vocab size: {len(vocab)}")


vocab size: 20521


### 3. Model 생성

05주차에서 만든 Encoder/Decoder를 각각 2층씩 쌓아 완전한 Encoder-Decoder Transformer를 만든다.
(GPU가 없어도 합리적인 시간 안에 학습이 끝나도록 작은 규모로 설정했다. GPU가 있다면 `d_model`, `num_layers`를 키워도 좋다.)


In [10]:
MAX_LEN = 30

config = dict(
    d_model=128,
    num_layers=2,
    num_heads=4,
    d_ff=256,
    max_len=MAX_LEN + 1,
    dropout=0.1,
)

model = ChatbotTransformer(vocab_size=len(vocab), pad_idx=stoi["<pad>"], **config).to(device)
model


ChatbotTransformer(
  (embed_layer): Embedding(20521, 128, padding_idx=0)
  (pe_layer): PositionalEncoding()
  (dropout): Dropout(p=0.1, inplace=False)
  (encoder): Encoder(
    (layers): ModuleList(
      (0-1): 2 x EncoderLayer(
        (mha): MultiHeadAttention(
          (W_q): Linear(in_features=128, out_features=128, bias=True)
          (W_k): Linear(in_features=128, out_features=128, bias=True)
          (W_v): Linear(in_features=128, out_features=128, bias=True)
          (W_o): Linear(in_features=128, out_features=128, bias=True)
        )
        (ffn): PositionwiseFeedForward(
          (net): Sequential(
            (0): Linear(in_features=128, out_features=256, bias=True)
            (1): ReLU()
            (2): Dropout(p=0.1, inplace=False)
            (3): Linear(in_features=256, out_features=128, bias=True)
          )
        )
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)


### 4. Dataset과 DataLoader 생성


In [11]:
train_dataset = ChatbotDataset(train_q, train_a, stoi, max_len=MAX_LEN)
test_dataset = ChatbotDataset(test_q, test_a, stoi, max_len=MAX_LEN)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

src, tgt_in, tgt_out = next(iter(train_dataloader))
src.shape, tgt_in.shape, tgt_out.shape


(torch.Size([16, 30]), torch.Size([16, 30]), torch.Size([16, 30]))

### 5. 학습

Encoder 입력(Q)과 Decoder 입력(`<sos>`+A)을 함께 넣어 다음 토큰(A+`<eos>`)을 맞히도록 학습한다(teacher forcing).
`<pad>` 위치는 `CrossEntropyLoss(ignore_index=pad_idx)`로 손실 계산에서 제외한다.

> Colab이라면 런타임 유형을 GPU로 바꾸면 훨씬 빠르다(런타임 > 런타임 유형 변경 > GPU). CPU에서는 20 epoch 기준 다소 시간이 걸릴 수 있으니, 급하면 `NUM_EPOCHS`를 줄여도 된다.


In [12]:
PAD_IDX = stoi["<pad>"]

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

LOG_INTERVAL = 50
NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):
    # 1. model train
    model.train()
    train_loss = 0.0

    with tqdm(train_dataloader, desc=f"[Epoch {epoch+1}/{NUM_EPOCHS}] Train", leave=False) as train_bar:
        for batch_idx, (src, tgt_in, tgt_out) in enumerate(train_bar):
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

            # 1-1. model 예측
            logits = model(src, tgt_in)  # [B, T, vocab_size]

            # 1-2. loss function 계산 ([B*T, vocab_size] vs [B*T])
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            # 1-3. 이전 기울기 초기화
            optimizer.zero_grad()

            # 1-4. 역전파(backpropagation) -> 기울기 계산
            loss.backward()

            # 1-5. 파라미터 업데이트
            optimizer.step()

            train_loss += loss.item() * src.size(0)

            if (batch_idx + 1) % LOG_INTERVAL == 0:
                train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_dataloader.dataset)

    # 2. model eval
    model.eval()
    eval_loss = 0.0

    with torch.no_grad(), tqdm(test_dataloader, desc=f"[Epoch {epoch+1}/{NUM_EPOCHS}] Eval", leave=False) as eval_bar:
        for src, tgt_in, tgt_out in eval_bar:
            src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)

            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            eval_loss += loss.item() * src.size(0)

    eval_loss /= len(test_dataloader.dataset)

    print(f"epoch {epoch+1}/{NUM_EPOCHS} | train_loss {train_loss:.4f} | eval_loss {eval_loss:.4f}")


[Epoch 1/20] Train:   0%|          | 0/665 [00:00<?, ?it/s]

[Epoch 1/20] Eval:   0%|          | 0/74 [00:00<?, ?it/s]

epoch 1/20 | train_loss 7.0078 | eval_loss 6.4971


[Epoch 2/20] Train:   0%|          | 0/665 [00:00<?, ?it/s]

[Epoch 2/20] Eval:   0%|          | 0/74 [00:00<?, ?it/s]

epoch 2/20 | train_loss 5.7231 | eval_loss 5.8505


... (50 outputs left out)

epoch 19/20 | train_loss 2.1254 | eval_loss 5.9500


[Epoch 20/20] Train:   0%|          | 0/665 [00:00<?, ?it/s]

[Epoch 20/20] Eval:   0%|          | 0/74 [00:00<?, ?it/s]

epoch 20/20 | train_loss 2.1064 | eval_loss 5.9453


In [13]:
checkpoint_path = "chatbot_transformer.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "stoi": stoi,
    "itos": itos,
    "config": config,
}, checkpoint_path)

print(f"모델 저장 완료: {checkpoint_path}")

모델 저장 완료: chatbot_transformer.pt


In [14]:
from google.colab import drive

drive.mount('/content/drive')

import shutil

shutil.copy('chatbot_transformer.pt', '/content/drive/MyDrive/chatbot_transformer.pt')
print("Drive에 복사 완료!")

Mounted at /content/drive
Drive에 복사 완료!


### 6. 예측 / 데모

학습된 모델로 몇 가지 질문에 답해보고, `exit`을 입력할 때까지 직접 대화도 해본다 (05주차 데모와 동일한 패턴).


In [ ]:
for q in ["안녕", "오늘 너무 피곤해", "밥 먹었어?"]:
    print(f"Q: {q}")
    print(f"A: {generate_reply(model, q, stoi, itos, max_len=MAX_LEN, device=device)}")
    print()

Q: 안녕
A: 잘 해결되길 바랄게요.

Q: 오늘 너무 피곤해
A: 잘 해결되길 바랄게요.

Q: 밥 먹었어?
A: 잘 살 수 있을 거예요.



In [ ]:
while True:
    text = input("나: ")
    if text == "exit":
        break

    reply = generate_reply(model, text, stoi, itos, max_len=MAX_LEN, device=device)
    print(f">> 챗봇: {reply}")

### 오늘 배운 것 & 다음 단계

* 완전한 Encoder-Decoder Transformer를 조립
* 실제 한국어 대화 데이터(Chatbot_data)로 seq2seq 학습 → 체크포인트(`chatbot_transformer.pt`) 저장
* 다음 단계: `final.py`를 `streamlit run final.py`로 실행하면, 이 체크포인트를 불러와 실시간 채팅 + 하드코딩 Function Calling(시간/날짜 조회) Agent로 사용할 수 있다.
